# Preprocessing pipeline

In [1]:
#imports

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import librosa.display
import os
import librosa as lb
import soundfile as sf

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# Loading data -> all factors dataframe

Demographic data
  - Patient number
  - Age
  - Sex
  - Adult BMI (kg/m2)
  - Child Weight (kg)
  - Child Height (cm)

Patient data

Audio text files

In [2]:
demographic_data = pd.read_csv('/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/demographic_info.txt',
                               sep=' ',
                               header=None,
                               names=["pid", "age", "sex", "adult_bmi", "child_weight", "child_height"])

print(demographic_data)

     pid    age  sex  adult_bmi  child_weight  child_height
0    101   3.00    F        NaN          19.0          99.0
1    102   0.75    F        NaN           9.8          73.0
2    103  70.00    F      33.00           NaN           NaN
3    104  70.00    F      28.47           NaN           NaN
4    105   7.00    F        NaN          32.0         135.0
..   ...    ...  ...        ...           ...           ...
121  222  60.00    M        NaN           NaN           NaN
122  223    NaN  NaN        NaN           NaN           NaN
123  224  10.00    F        NaN          32.3         143.0
124  225   0.83    M        NaN           7.8          74.0
125  226   4.00    M        NaN          16.7         103.0

[126 rows x 6 columns]


In [3]:
#load patient data to DF

patient_data=pd.read_csv('/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv',
                         names=['pid','disease'])
patient_data.head()

,pid,disease
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI


In [4]:
#load audio files

path='/Users/lilyeastwood/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
files=[s.split('.')[0] for s in os.listdir(path) if '.txt' in s]
files[:5]

print(f'Number of audio files: {len(files)}')

#build df of info - row per breath cycle

def getFilenameInfo(file):
    parts = file.split('_')
    return {
        'patient_number': parts[0],
        'recording_index': parts[1],
        'chest_location': parts[2],
        'acquisition_mode': parts[3],
        'equipment': parts[4]
    }

files_data=[]
for file in files:
    data=pd.read_csv(path + file + '.txt',sep='\t',names=['start','end','crackles','weezels'])
    name_data=getFilenameInfo(file)
    data['pid']=name_data['patient_number']
    data['recording_index']=name_data['recording_index']
    data['chest_location']=name_data['chest_location']
    data['acquisition_mode']=name_data['acquisition_mode']
    data['equipment']=name_data['equipment']
    data['filename']=file
    files_data.append(data)

files_df=pd.concat(files_data)
files_df.reset_index()
files_df.head()

Number of audio files: 920


,start,end,crackles,weezels,pid,recording_index,chest_location,acquisition_mode,equipment,filename
0,0.022,0.364,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron
1,0.364,2.436,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron
2,2.436,4.636,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron
3,4.636,6.793,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron
4,6.793,8.750,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron


In [5]:
#join patient info with text files

patient_data.pid=patient_data.pid.astype('int32')
files_df.pid=files_df.pid.astype('int32')

audio_data=pd.merge(files_df,patient_data,on='pid')
audio_data.head()

,start,end,crackles,weezels,pid,recording_index,chest_location,acquisition_mode,equipment,filename,disease
0,0.022,0.364,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron,URTI
1,0.364,2.436,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron,URTI
2,2.436,4.636,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron,URTI
3,4.636,6.793,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron,URTI
4,6.793,8.750,0,0,148,1b1,Al,sc,Meditron,148_1b1_Al_sc_Meditron,URTI


In [6]:
#combination dataframe

allfactors_data=pd.merge(audio_data,demographic_data,on='pid')
allfactors_data
allfactors_data.head()
allfactors_data['disease'].value_counts()

disease
COPD              5746
Healthy            322
Pneumonia          285
URTI               243
Bronchiolitis      160
Bronchiectasis     104
LRTI                32
Asthma               6
Name: count, dtype: int64

# Cleaning

-blanks

In [7]:
#block to complete

# Preprocessing
-encoding
-standardising

In [8]:
#dealing with numericals

#one hot encode disease, equipment, acquisition mode, chest location, recording_index

cols = ['disease', 'recording_index', 'chest_location', 'acquisition_mode']
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(allfactors_data[cols])

encoded_df = pd.DataFrame(encoded,
                          columns=encoder.get_feature_names_out(cols),
                          index=allfactors_data.index)

allfactors_data = pd.concat([allfactors_data, encoded_df], axis=1).drop(columns=cols)

#binary code gender
le = LabelEncoder()
allfactors_data['sex'] = allfactors_data['sex'].map({'M': 0, 'F': 1})
allfactors_data.head()


,start,end,crackles,weezels,pid,equipment,filename,age,sex,adult_bmi,...,recording_index_8p3,chest_location_Al,chest_location_Ar,chest_location_Ll,chest_location_Lr,chest_location_Pl,chest_location_Pr,chest_location_Tc,acquisition_mode_mc,acquisition_mode_sc
0,0.022,0.364,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.364,2.436,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2.436,4.636,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4.636,6.793,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,6.793,8.750,0,0,148,Meditron,148_1b1_Al_sc_Meditron,4.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


# Features engineering
-one BMI column

In [9]:
#one BMI column

def calculate_bmi(row):
    if row['age'] >= 19:  # adult
        return row['adult_bmi'] / 25 if pd.notna(row['adult_bmi']) else np.nan

    if pd.isna(row['child_weight']) or pd.isna(row['child_height']):
        return np.nan

    raw_bmi = row['child_weight'] / (row['child_height'] / 100) ** 2
    age = row['age']
    sex = row['sex']  # M=0, F=1

    if age < 2:
        return raw_bmi / 16.5
    elif sex == 0:  # Male
        thresholds = [(9, 16), (11, 17), (12, 18), (13, 18.6), (14, 19.3),
                      (15, 20), (16, 20.6), (17, 21.3), (18, 22), (19, 22.6), (20, 23)]
        for max_age, divisor in thresholds:
            if age <= max_age:
                return raw_bmi / divisor
    elif sex == 1:  # Female
        thresholds = [(9, 16), (11, 17.5), (12, 18), (13, 18.6), (14, 19.3),
                      (15, 20), (16, 20.5), (17, 20.8), (18, 21.2), (19, 21.5), (20, 21.8)]
        for max_age, divisor in thresholds:
            if age <= max_age:
                return raw_bmi / divisor

    return np.nan

allfactors_data['bmi'] = allfactors_data.apply(calculate_bmi, axis=1)

print(allfactors_data['bmi'].isnull().sum())
allfactors_data['bmi'].describe()

#length of breathing cycle
allfactors_data['cycle_length'] = allfactors_data['end']-allfactors_data['start']

#flip healthy column
allfactors_data['disease_Healthy'] = 1 - allfactors_data['disease_Healthy']
allfactors_data = allfactors_data.rename(columns={'disease_Healthy': 'disease_not_healthy'})

168
